In [5]:
# Step 1 - Hit the api 
import requests
import pandas as pd
from datetime import datetime
import os

In [10]:
headers = {'User-Agent':'Mozilla/5.0'}
response = requests.get('https://api.github.com/users/torvalds/repos',headers=headers)
print(response.status_code)
print(response.json())

# this pulls the public repository for a real user (torvalds=Linus Torvalds). No api key needed for public github data - just a user-agent header

200
[{'id': 940929652, 'node_id': 'R_kgDOOBVydA', 'name': '1590A', 'full_name': 'torvalds/1590A', 'private': False, 'owner': {'login': 'torvalds', 'id': 1024025, 'node_id': 'MDQ6VXNlcjEwMjQwMjU=', 'avatar_url': 'https://avatars.githubusercontent.com/u/1024025?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/torvalds', 'html_url': 'https://github.com/torvalds', 'followers_url': 'https://api.github.com/users/torvalds/followers', 'following_url': 'https://api.github.com/users/torvalds/following{/other_user}', 'gists_url': 'https://api.github.com/users/torvalds/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/torvalds/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/torvalds/subscriptions', 'organizations_url': 'https://api.github.com/users/torvalds/orgs', 'repos_url': 'https://api.github.com/users/torvalds/repos', 'events_url': 'https://api.github.com/users/torvalds/events{/privacy}', 'received_events_url': 'https://api.github.com/user

In [13]:
# Step 2 : Check and Parse

if response.status_code == 200:
    data = response.json()
    print(f"{len(data)} repos recieved")
else:
    print(f"Failed:{response.status_code}")

# data becomes a list of dictionaries - one dictionary per repository, each holding 81 fields of metadata

12 repos recieved


In [16]:
# Step 3 : JSON -> dataframe

df = pd.DataFrame(data)
print(df.shape)
print(df)

# pd.DataFrame turns the list of dicts directly into a table - each dict becomes a row and each key becomes a column

(12, 81)
            id                           node_id                 name  \
0    940929652                      R_kgDOOBVydA                1590A   
1   1130786764                      R_kgDOQ2ZvzA           AudioNoise   
2   1058343058                      R_kgDOPxUIkg          GuitarPedal   
3   1137038093                      R_kgDOQ8XTDQ     HunspellColorize   
4     79171906  MDEwOlJlcG9zaXRvcnk3OTE3MTkwNg==       libdc-for-dirk   
5    519408694                      R_kgDOHvWMNg              libgit2   
6      2325298      MDEwOlJlcG9zaXRvcnkyMzI1Mjk4                linux   
7    113099837  MDEwOlJlcG9zaXRvcnkxMTMwOTk4Mzc=           pesconvert   
8   1257356954                      R_kgDOSvG-mg          ScrollWheel   
9     78665021  MDEwOlJlcG9zaXRvcnk3ODY2NTAyMQ==  subsurface-for-dirk   
10    86106493  MDEwOlJlcG9zaXRvcnk4NjEwNjQ5Mw==             test-tlb   
11   117900805  MDEwOlJlcG9zaXRvcnkxMTc5MDA4MDU=               uemacs   

                       full_name  private

In [19]:
# Step 4 : Trim Columns

df_clean = df[["name","stargazers_count","language","forks_count"]].copy()
print(df_clean)

                   name  stargazers_count  language  forks_count
0                 1590A               573  OpenSCAD           21
1            AudioNoise              4489         C          219
2           GuitarPedal              2314         C          109
3      HunspellColorize               378         C           20
4        libdc-for-dirk               404         C           51
5               libgit2               387         C           30
6                 linux            249052         C        64548
7            pesconvert               577         C           74
8           ScrollWheel               373         C           12
9   subsurface-for-dirk               472       C++           67
10             test-tlb              1058         C          221
11               uemacs              2138         C          322


In [20]:
# Step 5 - Handle missing values

print(df_clean.isna().sum())
df_clean["language"] = df_clean['language'].fillna("Unknown")

name                0
stargazers_count    0
language            0
forks_count         0
dtype: int64


In [22]:
# Step 6 - Store the cleaned dataframe with a timestamp filename

timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
filename = f'github_repos_{timestamp}.csv'

df_clean.to_csv(filename,index=False)
print(f"Saved:{filename}")

Saved:github_repos_260915_161513.csv
